## Imports and Dataset

In [ ]:
from pathlib import Path

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [ ]:
RUN_ID = "6f701059-94c7-4969-8ca9-5ba2656995d9"
data_path = Path("../data/interim") / RUN_ID / f"validated_dataset_{RUN_ID}.parquet"

df = pd.read_parquet(data_path)

In [ ]:
df.sample(3)

## Row Filtering, Encoding & EDA

In [ ]:
good_loans = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid"
]

bad_loans = [
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off"
]

valid_status = good_loans + bad_loans

df_target = df[df["loan_status"].isin(valid_status)].copy()

target_map = {status: 0 for status in good_loans}
target_map.update({status: 1 for status in bad_loans})

df_target["target"] = df_target["loan_status"].map(target_map)

print("Filtered dataset shape:", df_target.shape)

#### EDA

In [ ]:
df_target["issue_d"] = pd.to_datetime(df_target["issue_d"])
df_target["issue_year"] = df_target["issue_d"].dt.year

df_target["issue_year"].describe()

In [ ]:
loan_vol_per_year = df_target["issue_year"].value_counts().sort_index()
loan_vol_per_year

In [ ]:
plt.figure(figsize=(10, 6))

loan_vol_per_year.plot(kind="bar")

plt.title("Loan Volume by Year")
plt.xlabel("Issue Year")
plt.ylabel("Number of Loans")

plt.show()

In [ ]:
default_rate_by_year = df_target.groupby("issue_year")["target"].mean()
default_rate_by_year

In [ ]:
plt.figure(figsize=(10,5))

default_rate_by_year.plot(marker="o")

plt.title("Default Rate by Loan Vintage")
plt.xlabel("Issue Year")
plt.ylabel("Default Rate")

plt.show()

In [ ]:
# Vintage Cohort Table
vintage_table = df_target.groupby("issue_year").agg(
    loans=("target", "count"),
    defaults=("target", "sum"),
    default_rate=("target", "mean")
)

vintage_table

In [ ]:
train_years = list(range(2007, 2016))
validation_year = 2016
test_years = [2017, 2018]

print("Train years:", train_years)
print("Validation year:", validation_year)
print("Test years:", test_years)

## Dataset Split Analysis

In [ ]:
def dataset_split(year):
    if year in train_years:
        return "train"
    elif year == validation_year:
        return "validation"
    elif year in test_years:
        return "test"
    else:
        return "excluded"
    
df_target["dataset_split"] = df_target["issue_year"].apply(dataset_split)
df_target["dataset_split"].value_counts()

In [ ]:
df_split = df_target[df_target["dataset_split"] != "excluded"].copy()
df_split["dataset_split"].value_counts()

In [ ]:
split_summary = df_split.groupby("dataset_split").agg(
    rows=("target", "count"),
    defaults=("target", "sum"),
    default_rate=("target", "mean")
)

split_summary

In [ ]:
plt.figure(figsize=(8, 4))

sns.countplot(data=df_split, x="dataset_split")

plt.title("Dataset Split Distribution")

plt.show()

In [ ]:
pd.crosstab(df_split["issue_year"], df_split["dataset_split"])

In [ ]:
print("Final dataset shape:", df_split.shape)

In [ ]:
df_split.sample(3)